# Notion-Zotero Analysis

Full pipeline: load canonical bundles → build summary tables → clean → explore.

**Data sources** (pick one):
-  — bundles pulled via 
-  — bundles pulled via 
-  — bundles parsed from local fixture files

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_style("whitegrid")

from notion_zotero.analysis import (
    load_canonical_records,
    build_summary_dataframes,
    clean_table,
    run_analysis,
    is_accepted,
    task_label_fn,
)
from notion_zotero.analysis.original_db_summary import (
    TYPO_FIXES,
    GENERIC_VALUE_MAP,
    SEARCH_STRATEGY_COLUMNS,
)

accepted_only = True  # Set to True to analyze only accepted papers, False to include all

# Load from live pulled data
pulled_dir = Path("data/pulled/notion/learning_analytics_review")
if not pulled_dir.exists() or not any(pulled_dir.glob("*.canonical.json")):
    raise FileNotFoundError(
        f"No pulled data found in {pulled_dir}. "
        "Run: notion-zotero pull-notion"
    )

bundles = load_canonical_records(str(pulled_dir))
print(f"Loaded {len(bundles)} bundles from {pulled_dir}")

In [ ]:
# --- Cell 2: Inspect loaded bundles ---
print(f"Loaded {len(bundles)} canonical bundles")

if bundles:
    sample = bundles[0]
    refs = sample.get("references", [])
    if refs:
        r = refs[0]
        print(f"Sample: {r.get("title", "(no title)")} ({r.get("year", "?")})")

In [ ]:
# --- Cell 3: Build summary tables (accepted papers only) ---
if accepted_only is True:
    accepted_bundles = [b for b in bundles if is_accepted(b)]
    print(f"Accepted: {len(accepted_bundles)} / {len(bundles)} total bundles")

else:
    accepted_bundles = bundles
    print(f"Including all bundles (accepted + excluded): {len(accepted_bundles)} total bundles")

dfs = build_summary_dataframes(accepted_bundles, task_label_fn=task_label_fn)
for name, df in dfs.items():
    print(f"  {name}: {len(df)} rows x {len(df.columns)} cols")
dfs.get("Reading List")

In [ ]:
# --- Cell 4: Clean tables ---
# MY_TYPO_FIXES and MY_VALUE_MAP extend the package defaults with notebook-specific overrides.
MY_TYPO_FIXES = {
    **TYPO_FIXES,
    r"\bMAchine\b": "Machine",
    r"Fiiltering": "Filtering",
    r"Exerrcise": "Exercise",
}
MY_VALUE_MAP = {**GENERIC_VALUE_MAP, "n/a": "Not Applicable", "none": "Not Applicable"}

cleaned_dfs = {}
_clean_logs = {}
for name, df in dfs.items():
    cleaned, log = clean_table(
        df,
        typo_fixes=MY_TYPO_FIXES,
        value_map=MY_VALUE_MAP,
        search_strategy_columns=SEARCH_STRATEGY_COLUMNS,
    )
    cleaned_dfs[name] = cleaned
    _clean_logs[name] = log
    print(f"  {name}: {log['text_updates']} cell(s) normalised")

In [ ]:
norm_log = {
    name: {
        "rows_before": len(dfs[name]),
        "rows_after": len(cleaned_dfs[name]),
        "text_updates": _clean_logs[name]["text_updates"],
        "search_strategy_updates": _clean_logs[name].get("search_strategy_updates", 0),
    }
    for name in dfs
}
pd.DataFrame(norm_log).T

In [ ]:
rl = cleaned_dfs.get("Reading List", pd.DataFrame())
if not rl.empty and "year" in rl.columns:
    display(rl["year"].value_counts().sort_index())
if not rl.empty and "journal" in rl.columns:
    display(rl["journal"].dropna().replace("", pd.NA).dropna().value_counts().head(10))
if not rl.empty and "database" in rl.columns:
    display(rl["database"].dropna().replace("", pd.NA).dropna().value_counts().head(10))
if not rl.empty and "journal_quartile" in rl.columns:
    display(rl["journal_quartile"].dropna().replace("", pd.NA).dropna().value_counts())
for name, df in cleaned_dfs.items():
    if name != "Reading List" and not df.empty:
        print(f"\n{name} ({len(df)} rows):")
        display(df.head(3))

## Paper Analysis Outputs

1. Check and count Unique Search Terms

In [ ]:
# #get value_counts of unique search terms from Reading List
# if "Reading List" in cleaned_dfs:
#     search_terms = cleaned_dfs["Reading List"]["search_terms"].dropna().replace("", pd.NA).dropna()
#     print("\nUnique search terms and their counts:")
#     display(search_terms.value_counts())

2. Get Rejection Status - may also need some typo correction and text normalization, similar to Search Terms Column

In [ ]:
#group by Status and Motive For Exclusion and count
reading_list = cleaned_dfs.get("Reading List", pd.DataFrame())
if not reading_list.empty and "Status" in reading_list.columns and "Motive For Exclusion" in reading_list.columns:
    exclusion_counts = (
        reading_list.groupby(["Status", "Motive For Exclusion"])
        .size()
        .reset_index(name="Count")
    )
    print("\nCounts by Status and Motive For Exclusion:")
    display(exclusion_counts)

3. Figure 2

Evolution of Data Representation Forms over the Years. Line and facet grid per task

In [ ]:
cleaned_dfs.get("Reading List", pd.DataFrame()).columns

In [ ]:
# Evolution of Data Representation Forms over the Years (Line Plot)

# Use the cleaned Reading List DataFrame
rl = cleaned_dfs.get("Reading List", pd.DataFrame())

# Check for expected columns
form_col = "Learner Representation"


# Clean and filter data
plot_df = rl[["year", form_col]].dropna()
plot_df = plot_df[plot_df["year"].astype(str).str.isnumeric()]
plot_df["year"] = plot_df["year"].astype(int)

# Count papers per year and form
counts = plot_df.groupby(["year", form_col]).size().reset_index(name="count")

plt.figure(figsize=(10, 6))
sns.lineplot(data=counts, x="year", y="count", hue=form_col, marker="o")
plt.title("Evolution of Data Representation Forms over the Years")
plt.ylabel("Number of Papers")
plt.xlabel("Year")
plt.legend(title="Form of Data Representation", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 4-pane Figure: Evolution of Data Representation Forms by Task (Modern Journal Style)
# Modern, journal-style settings
sns.set_theme(style="whitegrid", context="talk", font_scale=1.1)
plt.rcParams.update({
    "axes.titlesize": 18,
    "axes.labelsize": 15,
    "legend.fontsize": 13,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "axes.titleweight": "bold",
    "axes.labelweight": "bold",
    "figure.dpi": 120,
    "axes.edgecolor": "#333333",
    "axes.linewidth": 1.2,
    "legend.frameon": True,
    "legend.framealpha": 0.95,
    "legend.fancybox": True,
    "legend.edgecolor": "#333333",
})

rl = cleaned_dfs.get("Reading List", pd.DataFrame())
form_col = "Learner Representation"

# Explode list values in Learner Representation and remove 'Not Applicable'
rl = rl.explode(form_col)
rl = rl[rl[form_col].str.lower() != "not applicable"]

# Use Status and Status_1, filter to those starting with 'accepted for'
def get_task(row):
    for col in ["Status", "Status_1"]:
        val = row.get(col, None)
        if isinstance(val, str) and val.lower().startswith("accepted for"):
            return val
    return None

rl["Task"] = rl.apply(get_task, axis=1)
facet_df = rl[["year", form_col, "Task"]].dropna()
facet_df = facet_df[facet_df["year"].astype(str).str.isnumeric()]
facet_df["year"] = facet_df["year"].astype(int)

# Optionally, select top 4 tasks by count
top_tasks = facet_df["Task"].value_counts().nlargest(4).index
facet_df = facet_df[facet_df["Task"].isin(top_tasks)]

# Count papers per year, form, and task
facet_counts = facet_df.groupby(["year", form_col, "Task"]).size().reset_index(name="count")

# Facet plot (sharey=True for same scale)
g = sns.relplot(
    data=facet_counts,
    x="year", y="count", hue=form_col,
    col="Task", kind="line", marker="o",
    col_wrap=2, height=4, aspect=1.3, facet_kws={'sharey': True},
    palette="Set2", linewidth=2.5, alpha=0.95
)
g.set_titles("Task: {col_name}")
g.set_axis_labels("Year", "Number of Papers")
g._legend.set_title("Learner Representation")
g._legend.set_bbox_to_anchor((0.5, -0.15))
g._legend.set_loc("lower center")
g._legend.set_frame_on(True)
g.fig.subplots_adjust(bottom=0.22)
g.fig.suptitle("Evolution of Data Representation Forms by Task (Top 4 Tasks)", y=1.04, fontsize=20, fontweight="bold")
sns.despine()
plt.show()

In [ ]:
# Inspect columns in the Reading List table
df = cleaned_dfs.get("Reading List", pd.DataFrame())
display(pd.DataFrame({"Column Name": df.columns}))
df.head(3)